In [ ]:
from datascience import *
import numpy as np

%matplotlib inline
import matplotlib.pyplot as plots
plots.style.use('fivethirtyeight')

## Functions

In [ ]:
def standard_units(x):
    return (x - np.mean(x))/np.std(x)

def distance(pt1, pt2):
    """Return the distance between two points, represented as arrays"""
    return np.sqrt(sum((pt1 - pt2)**2))

def row_distance(row1, row2):
    """Return the distance between two numerical rows of a table"""
    return distance(np.array(row1), np.array(row2))

def distances(training, example):
    """
    Compute distance between example and every row in training.
    Return training augmented with Distance column
    """
    distances = make_array()
    attributes_only = training.drop('Class')

    for i in np.arange(attributes_only.num_rows):
        row = attributes_only.row(i)
        distances = np.append(distances, row_distance(row, example))
        
    return training.with_column('Distance from Example', distances)

def classify(training, example, k):
    """
    Return the majority class among the k nearest neighbors of example
    """
    # table with closest k rows to example
    closest_k = distances(training, example
                         ).sort('Distance from Example').take(np.arange(k))

    # grouped table of classes and counts for closest k rows    
    grouped_table = closest_k.group('Class')

    # majority class for closest k rows
    majority_class = grouped_table.sort('count', descending = True
                                       ).column(0).item(0)
    
    return majority_class

def classify_all(training, test, k):
    """Return an array of predicted classes for every row in test"""
    test_attributes = test.drop('Class')

    predictions = make_array()

    for i in np.arange(test_attributes.num_rows):
        row = test_attributes.row(i)
        predictions = np.append(predictions, classify(training, row, k))
        
    return predictions

def evaluate_accuracy(training, test, k):
    """Return the proportion of correctly classified examples in the test set"""
    predictions = classify_all(training, test, k)
    
    return np.mean(predictions == test.column('Class'))

def evaluate_accuracy_by_class(training, test, k):
    """Return the proportion of correctly classified examples in the test set for each class"""
    predictions = classify_all(training, test, k)
    results = test.with_column(
        'Correct',
        predictions == test.column('Class'))

    class_0 = results.where('Class', 0).column('Correct')
    class_1 = results.where('Class', 1).column('Correct')
    
    return Table().with_columns('Class 0 Accuracy', np.mean(class_0),
                                'Class 1 Accuracy', np.mean(class_1))


## Counterfeit Currency

In [ ]:
banknotes = Table.read_table('/home/jovyan/shared/Data/banknote.csv')

# standardize data

banknotes_SU = Table().with_columns(
    'WaveletVar SU', standard_units(banknotes.column('WaveletVar')),
    'WaveletSkew SU', standard_units(banknotes.column('WaveletSkew')),
    'WaveletCurt SU', standard_units(banknotes.column('WaveletCurt')),
    'Class', banknotes.column('Class'))

# Split into training and testing sets

train_size = int(np.round(banknotes.num_rows * 0.80))

np.random.seed(2023)

shuffled_banknotes_SU = banknotes_SU.sample(with_replacement=False) 
banknotes_SU_train = shuffled_banknotes_SU.take(np.arange(train_size))
banknotes_SU_test  = shuffled_banknotes_SU.take(np.arange(train_size, banknotes.num_rows))

In [ ]:
evaluate_accuracy(banknotes_SU_train, banknotes_SU_test, 9)

In [ ]:
accuracy_table = evaluate_accuracy_by_class(banknotes_SU_train, banknotes_SU_test, 9)
accuracy_table

### If we test a random bill and classify it as counterfeit, what is the probability that it is _actually_ counterfeit?

According to the United States Department of Treasury, an estimated 30 million dollars in counterfeit bills are in circulation, or approximately 1 note in counterfeits for every 40,000 in genuine currency ([source](https://www.federalreserve.gov/econres/ifdp/estimating-the-volume-of-counterfeit-us-currency-in-circulation.htm)).

P(counterfeit) =  
P(Not counterfeit) = 


P(test+ | counterfeit) =    
P(test- | counterfeit) =     

P(test+ | not counterfeit) =    
P(test- | not counterfeit) =   

In [ ]:
class_0_accuracy = accuracy_table.column(0).item(0)
class_1_accuracy = accuracy_table.column(1).item(0)

class_0_accuracy, class_1_accuracy

### If we test a random bill and classify it as _not_ counterfeit, what is the probability that it is _actually_ counterfeit?

## Breast Cancer Example

Breast abnormalities can also be found through self-examination or when symptoms arise. Suspicious mammographic findings may require a biopsy for diagnosis. More than 1 million women have breast biopsies each year in the United States. About 20 percent of these biopsies yield a diagnosis of breast cancer ([source](https://effectivehealthcare.ahrq.gov/products/breast-biopsy-update/clinician#:~:text=Breast%20abnormalities%20can%20also%20be,a%20diagnosis%20of%20breast%20cancer.)).

In [ ]:
patients = Table.read_table('/home/jovyan/shared/Data/breast-cancer.csv').drop('ID')

train_size = int(np.round(patients.num_rows * 0.8))

np.random.seed(120003)

shuffled = patients.sample(with_replacement=False) 
training_set = shuffled.take(np.arange(train_size))
test_set  = shuffled.take(np.arange(train_size, patients.num_rows))

accuracy_table_bc = evaluate_accuracy_by_class(training_set, test_set, 11)
accuracy_table_bc

### If we test a tissue sample and classify it as cancer, what is the probability that it is _actually_ cancer?

P(cancer) =     
P(Not cancer) = 

P(test+ | cancer) =    
P(test- | cancer) =    

P(test+ | not cancer) =    
P(test- | not cancer) = 

In [ ]:
class_0_accuracy_bc = accuracy_table_bc.column(0).item(0)
class_1_accuracy_bc = accuracy_table_bc.column(1).item(0)

### If we test a tissue sample and classify it as _not_ cancer, what is the probability that it is _not_ cancer?